# RAG Pipeline — End-to-End Test

In [1]:
import sys
sys.path.insert(0, '..')
from pathlib import Path

from src.document_loader import DocumnetLoader

loader = DocumnetLoader()

documents = loader.load_documents(Path("../data/raw"))

documents[0]
print(type(documents))
print(len(documents))

<class 'list'>
3


In [2]:
print(documents[0])
print(type(documents[0]))
print(documents[0].keys())

{'text': 'Retrieval-Augmented Generation (RAG) is a technique that enhances LLMs by retrieving\nrelevant documents from an external knowledge base before generating a response.\n\nRAG Pipeline:\n1. Document Loading: read raw documents from files or databases\n2. Chunking: split documents into smaller overlapping chunks\n3. Embedding: convert chunks into dense vector representations\n4. Vector Store: index and store embeddings for fast similarity search\n5. Retrieval: given a query, find the top-k most similar chunks\n6. Generation: pass retrieved chunks + query to LLM for final answer\n\nAdvantages of RAG:\n- Reduces hallucination by grounding responses in real documents\n- Knowledge can be updated without retraining the model\n- Provides citations and source traceability\n- Works well with domain-specific or private data\n\nCommon embedding models: sentence-transformers, OpenAI ada-002\nCommon vector stores: FAISS, Chroma, Pinecone, Weaviate\n', 'source': 'rag_intro.txt'}
<class 'dict

In [3]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
from src.chunker import Chunker

chunker = Chunker()

chunks = chunker.chunk_documents(documents)

print(type(chunks))
print(len(chunks))
print(chunks[0])

<class 'list'>
11
{'text': 'Retrieval-Augmented Generation (RAG) is a technique that enhances LLMs by retrieving\nrelevant documents from an external knowledge base before generating a response.', 'source': 'rag_intro.txt', 'chunk_id': 0}


In [4]:
from src.embedder import EmbeddingGenerator

embedder = EmbeddingGenerator()
embedded_chunks  = embedder.embed_chunks(chunks)

print(type(embedded_chunks))
print(len(embedded_chunks))
print(embedded_chunks[0].keys())
print(type(embedded_chunks[0]["embedding"]))
print(len(embedded_chunks[0]["embedding"]))

/Users/hojjatkamali/Tamrin1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'list'>
11
dict_keys(['text', 'source', 'chunk_id', 'embedding'])
<class 'list'>
384


In [5]:
from src.embedder import EmbeddingGenerator
embedder = EmbeddingGenerator()

query_embedding = embedder.embed_query("What is RAG?")

print(type(query_embedding))
print(len(query_embedding))

<class 'list'>
384


In [6]:
from src.vector_store import VectorStore

store = VectorStore()
store.add_embeddings(embedded_chunks)

print(store.index)
print(len(store.metadata))


<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x17b5799b0> >
11


In [7]:
from src.vector_store import VectorStore

store = VectorStore()
store.add_embeddings(embedded_chunks)

results = store.search(query_embedding, top_k=3)

print(type(results))
print(len(results))
print(results[0].keys())
print(results[0])

<class 'list'>
3
dict_keys(['text', 'source', 'chunk_id', 'score'])
{'text': 'Retrieval-Augmented Generation (RAG) is a technique that enhances LLMs by retrieving\nrelevant documents from an external knowledge base before generating a response.', 'source': 'rag_intro.txt', 'chunk_id': 0, 'score': 0.960932731628418}


In [8]:
from src.rag_engine import RAGEngine
from src.text_generator import TextGenerator

generator = TextGenerator()
rag = RAGEngine(embedder, store, generator=generator)


result = rag.answer("What is RAG?")

print(result.keys())
print(result["question"])
print(result["sources"])
print(result["context"])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


dict_keys(['question', 'context', 'prompt', 'sources', 'retrieved_chunks', 'answer'])
What is RAG?
['rag_intro.txt', 'rag_intro.txt', 'rag_intro.txt']
Retrieval-Augmented Generation (RAG) is a technique that enhances LLMs by retrieving
relevant documents from an external knowledge base before generating a response.

Advantages of RAG:
- Reduces hallucination by grounding responses in real documents
- Knowledge can be updated without retraining the model
- Provides citations and source traceability
- Works well with domain-specific or private data

RAG Pipeline:
1. Document Loading: read raw documents from files or databases
2. Chunking: split documents into smaller overlapping chunks
3. Embedding: convert chunks into dense vector representations
4. Vector Store: index and store embeddings for fast similarity search
5. Retrieval: given a query, find the top-k most similar chunks
6. Generation: pass retrieved chunks + query to LLM for final answer


In [9]:
from src.rag_engine import RAGEngine
rag = RAGEngine(embedder, store,generator=generator)
result = rag.answer("What is RAG?")
print(result["prompt"])


        Use the following context to answer the question.
        
       Context:
       Retrieval-Augmented Generation (RAG) is a technique that enhances LLMs by retrieving
relevant documents from an external knowledge base before generating a response.

Advantages of RAG:
- Reduces hallucination by grounding responses in real documents
- Knowledge can be updated without retraining the model
- Provides citations and source traceability
- Works well with domain-specific or private data

RAG Pipeline:
1. Document Loading: read raw documents from files or databases
2. Chunking: split documents into smaller overlapping chunks
3. Embedding: convert chunks into dense vector representations
4. Vector Store: index and store embeddings for fast similarity search
5. Retrieval: given a query, find the top-k most similar chunks
6. Generation: pass retrieved chunks + query to LLM for final answer

       Question:
       What is RAG?

       Answer:
       


In [10]:
from src.text_generator import TextGenerator
generator = TextGenerator()

answer = generator.generate(
    "What is Retrieval-Augmented Generation?"
)

print(answer)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Retrieval-Augmented Generation


In [11]:
from src.text_generator import TextGenerator
generator = TextGenerator()

rag = RAGEngine(
    embedder=embedder,
    vector_store= store,
    generator = generator
)

result = rag.answer("what is RAG?")
print(result['answer'])
print(result['sources'])



Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


a technique that enhances LLMs by retrieving relevant documents from an external knowledge base before generating a response
['rag_intro.txt', 'rag_intro.txt', 'rag_intro.txt']


In [12]:
from src.document_loader import DocumnetLoader
from src.chunker import Chunker
from src.embedder import EmbeddingGenerator
from src.vector_store import VectorStore
from src.text_generator import TextGenerator
from src.rag_engine import RAGEngine
from pathlib import Path

Loader = DocumnetLoader()
documents = Loader.load_documents(Path("../data/raw"))


chunker = Chunker()
chunks = chunker.chunk_documents(documents)

embedder = EmbeddingGenerator()
embedded_chunks = embedder.embed_chunks(chunks)

store = VectorStore()
store.add_embeddings(embedded_chunks)

generator = TextGenerator()

rag = RAGEngine(
    embedder=embedder,

    vector_store=store,

    generator=generator
)

result = rag.answer("What is RAG?")

print(result['answer'])
print(result['sources'])


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


a technique that enhances LLMs by retrieving relevant documents from an external knowledge base before generating a response
['rag_intro.txt', 'rag_intro.txt', 'rag_intro.txt']


In [15]:
from pathlib import Path
from src.document_loader import DocumnetLoader
from src.chunker import Chunker
from src.embedder import EmbeddingGenerator
from src.vector_store import VectorStore

loader = DocumnetLoader()
docs = loader.load_documents(Path("../data/raw"))
chunks = Chunker().chunk_documents(docs)
embedded = EmbeddingGenerator().embed_chunks(chunks)

store = VectorStore()
store.add_embeddings(embedded)        # بار اول — ۱۱ چانک
store.add_embeddings(embedded[:5])    # بار دوم — شبیه‌سازی ۵ چانک جدید

print("metadata:", len(store.metadata))
assert store.index 
print("index:", store.index.ntotal)

metadata: 16
index: 16
